# GMM imputation sensitivity
Needs to be run after `imputation.R`, `dimensionality_reduction.R`, and `gmm_clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
import subprocess
from typing import TypedDict

import numpy as np
import pandas as pd
from numpy.typing import NDArray
from sklearn.base import clone
from sklearn.metrics import adjusted_rand_score as unweighted_ari

from utils.gmm_utils import GaussianMixture
from utils.metrics_utils import adjusted_rand_score as weighted_ari
from utils.saving_utils import load_object, save_object

In [ ]:
# Set the number of alternative imputations and paths to matching reference files
n_imputations = 10
seed = 346396
gmm_seed = 363453

imputation_folder = Path("data/imputed")
reference_mfa_folder = Path("data/dim_red")
reference_gmm_folder = Path("outputs/gmm_clustering")
output_folder = Path("outputs/gmm_clustering/imputation_sensitivity")

output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the reference models
class ReferenceClustering(TypedDict):
    model: GaussianMixture
    scores: pd.DataFrame
    labels: NDArray[np.int64]


reference: dict[str, ReferenceClustering] = {}
for wave in ("discov", "valid"):
    model = load_object(reference_gmm_folder / f"model_{wave}.pkl")
    scores = pd.read_parquet(reference_mfa_folder / f"data_{wave}_mfa.parquet")
    reference[wave] = {"model": model, "scores": scores, "labels": np.asarray(model.predict(scores), dtype=np.int64)}

In [ ]:
# Run MICE with the reference specifications and fit a MFA for each imputed dataset using the R script
subprocess.run(
    [
        "Rscript",
        str(project_path / "utils/prepare_gmm_imputation_sensitivity.R"),
        str(n_imputations),
        str(seed),
        str(reference["discov"]["model"].n_features_in_),
        str(reference["valid"]["model"].n_features_in_),
        str(imputation_folder),
        str(output_folder),
    ],
    cwd=project_path,
    check=True,
)

In [ ]:
# Refit each GMM with the saved parameters and compare partitions using ARI
comparison_rows: list[dict[str, int | float | str | bool]] = []
for wave, reference_data in reference.items():
    reference_model = reference_data["model"]
    reference_labels = reference_data["labels"]
    for imputation in range(1, n_imputations + 1):
        # Load alternative run after imputation and MFA
        path = output_folder / f"mfa_{wave}_{imputation:03d}.parquet"
        alternative_data = pd.read_parquet(path)
        sample_weight = alternative_data["sample_weight"].to_numpy(dtype=np.float64)
        alternative_scores = alternative_data.drop(columns="sample_weight")

        # Fit model and predict results
        fit_seed = np.random.SeedSequence([gmm_seed, imputation, wave == "valid"]).generate_state(1)[0]
        alternative_model = clone(reference_model).set_params(random_state=np.random.RandomState(fit_seed))
        alternative_model.fit(alternative_scores, sample_weight=sample_weight)
        alternative_labels = np.asarray(alternative_model.predict(alternative_scores), dtype=np.int64)

        comparison_rows.append(
            {
                "wave": wave,
                "imputation": imputation,
                "n_components": reference_model.n_components,
                "gmm_seed": fit_seed,
                "converged": bool(alternative_model.converged_),
                "ari": float(unweighted_ari(reference_labels, alternative_labels)),
                "weighted_ari": float(weighted_ari(reference_labels, alternative_labels, sample_weight=sample_weight)),
            }
        )

        pd.DataFrame({"reference_label": reference_labels, "alternative_label": alternative_labels}).to_parquet(
            output_folder / f"predictions_{wave}_{imputation:03d}.parquet", index=False
        )
        save_object(alternative_model, output_folder / f"model_{wave}_{imputation:03d}.pkl")

In [ ]:
# Save and inspect the comparison summaries
comparison = pd.DataFrame(comparison_rows)
comparison.to_parquet(output_folder / "comparison.parquet", index=False)
comparison